In [1]:
!pip install category_encoders

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import category_encoders as ce
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
import pandas as pd

df_4_6 = pd.read_csv(path_to_gold_data)
df_7_9 = pd.read_csv(path_to_gold_data)

In [4]:
df_4_6.columns

Index(['Order date', 'CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'INNER_CD',
       'SUPPLIER_CD', 'Sales order line number',
       'Consider count hodiday Saturday', 'SO QTY', 'OTHER AREA SHIP DIV',
       'SUPPLIER INV AMOUNT', 'PACKING RANK', 'LOGICAL PLANT', 'VSD',
       'DIRECT SHIP FLG', 'DELI_DIV', 'label', 'Ship Mode', 'PACK QTY',
       'WEIGHT PER PIECE', 'SUPPLIER_DIV', 'SPECIAL_DIV', 'SO_DAY_OF_MONTH',
       'SO_DAY_OF_WEEK', 'SO_TIME', 'SUPPLIER_CATEGORY_CD', 'SO_TIME_str',
       'hour', 'time_period', 'IS_WEEKEND', 'MONTH_PHASE', 'Order_month',
       'VSD_month', 'VSD_IS_WEEKEND', 'VSD_MONTH_PHASE',
       'Expected_delivery_days'],
      dtype='object')

'CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'INNER_CD',
       'SUPPLIER_CD' - Hash encode

In [5]:
# Select columns have dtype = object
object_cols = df_4_6.select_dtypes(include=['object']).columns.tolist()

print("Object columns:")
print(object_cols)

Object columns:
['Order date', 'BRAND_CD', 'INNER_CD', 'SUPPLIER_CD', 'PACKING RANK', 'VSD', 'DELI_DIV', 'Ship Mode', 'time_period', 'MONTH_PHASE', 'VSD_MONTH_PHASE']


# Chia theo train - dev - test

In [6]:
# Chia tập A
X_A = df_4_6.drop(columns=['label','Order date', 'VSD'])
y_A = df_4_6['label']

X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(X_A, y_A, test_size=0.2, random_state=42, stratify=y_A)

In [7]:
X_B = df_7_9.drop(columns=['label','Order date', 'VSD'])
y_B = df_7_9['label']

X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(X_B, y_B, test_size=0.2, random_state=42, stratify=y_B)

In [8]:
target_encode = ['CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'SUPPLIER_CD']
one_hot_encode = ['Consider count hodiday Saturday', 'OTHER AREA SHIP DIV','PACKING RANK', 'DIRECT SHIP FLG',
                  'DELI_DIV','LOGICAL PLANT','Ship Mode', 'WEIGHT PER PIECE','SUPPLIER_DIV',
                  'SPECIAL_DIV','time_period','IS_WEEKEND','MONTH_PHASE',
                  'Order_month','VSD_month','VSD_IS_WEEKEND','VSD_MONTH_PHASE']

In [9]:
preprocessor_A = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

preprocessor_B = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

# Undersampling majority (majority : minority = 4 : 1)
undersample = RandomUnderSampler(
    sampling_strategy=0.5,
    random_state=42
)
smote = SMOTE(sampling_strategy=0.25, random_state=42 )
pipeline_A = ImbPipeline(steps=[
    ('preprocess', preprocessor_A),
    ('smote', smote),
    ('undersample', undersample),
])

pipeline_B = ImbPipeline(steps=[
    ('preprocess', preprocessor_B),
    ('smote', smote),
    ('undersample', undersample),
])
X_A_resampled, y_A_resampled = pipeline_A.fit_resample(X_A_train, y_A_train)
X_B_resampled, y_B_resampled = pipeline_B.fit_resample(X_B_train, y_B_train)
print("Before sampling:")
print(y_A_train.value_counts())

print("\nAfter sampling:")
print(pd.Series(y_A_resampled).value_counts())


Before sampling:
label
0    311456
1      7786
Name: count, dtype: int64

After sampling:
label
0    155728
1     77864
Name: count, dtype: int64


In [10]:
X_A_test_processed = pipeline_A.named_steps['preprocess'].transform(X_A_test)
X_B_test_processed = pipeline_B.named_steps['preprocess'].transform(X_B_test)

In [11]:
X_B_test_processed1 = pipeline_A.named_steps['preprocess'].transform(X_B_test)
X_A_test_processed1 = pipeline_B.named_steps['preprocess'].transform(X_A_test)

In [12]:
neg = y_A_resampled.value_counts()[0]
pos = y_A_resampled.value_counts()[1]

In [13]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_A_resampled, y_A_resampled)

y_A_pred_knn = knn.predict(X_A_test_processed)
y_A_prob_knn = knn.predict_proba(X_A_test_processed)[:, 1]

print("==== KNN ====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_knn))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_knn, digits=4))

auc_knn = roc_auc_score(y_A_test, y_A_prob_knn)
print(f"AUC-ROC: {auc_knn:.4f}")

==== KNN ====
Confusion Matrix:
[[64575 13289]
 [  313  1634]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9952    0.8293    0.9047     77864
           1     0.1095    0.8392    0.1937      1947

    accuracy                         0.8296     79811
   macro avg     0.5523    0.8343    0.5492     79811
weighted avg     0.9736    0.8296    0.8874     79811

AUC-ROC: 0.8749


In [14]:
neg = y_A_train.value_counts()[0]
pos = y_A_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric='aucpr',
    scale_pos_weight=neg/pos
)

xgb_model.fit(X_A_resampled, y_A_resampled)

y_A_pred_xgb= xgb_model.predict(X_A_test_processed)
y_A_prob_xgb = xgb_model.predict_proba(X_A_test_processed)[:, 1]

print("==== XGBoost ====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_xgb, digits=4))

auc_xgb = roc_auc_score(y_A_test, y_A_prob_xgb)
print(f"AUC-ROC: {auc_xgb:.4f}")

==== XGBoost ====
Confusion Matrix:
[[64685 13179]
 [  237  1710]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9963    0.8307    0.9060     77864
           1     0.1148    0.8783    0.2031      1947

    accuracy                         0.8319     79811
   macro avg     0.5556    0.8545    0.5546     79811
weighted avg     0.9748    0.8319    0.8889     79811

AUC-ROC: 0.9320


In [15]:
lgbm_model = LGBMClassifier(
    objective='binary',
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=35,
    random_state=42,
    scale_pos_weight=neg/pos
)
lgbm_model.fit(X_A_resampled, y_A_resampled)

y_A_pred_lgbm = lgbm_model.predict(X_A_test_processed)
y_A_prob_lgbm = lgbm_model.predict_proba(X_A_test_processed)[:, 1]

print("==== LightGBM====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_lgbm))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_lgbm, digits=4))

auc_lgbm = roc_auc_score(y_A_test, y_A_prob_lgbm)
print(f"AUC-ROC: {auc_lgbm:.4f}")

[LightGBM] [Info] Number of positive: 77864, number of negative: 155728
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.682925 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41378
[LightGBM] [Info] Number of data points in the train set: 233592, number of used features: 1080
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


==== LightGBM====
Confusion Matrix:
[[68014  9850]
 [  276  1671]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9960    0.8735    0.9307     77864
           1     0.1450    0.8582    0.2481      1947

    accuracy                         0.8731     79811
   macro avg     0.5705    0.8659    0.5894     79811
weighted avg     0.9752    0.8731    0.9141     79811

AUC-ROC: 0.9444


# Chia theo K-Fold

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, make_scorer

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9]
}

f1_scorer = make_scorer(f1_score, pos_label=1)

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for KNN")
grid_knn.fit(X_A_resampled, y_A_resampled)

print("Best parameters (KNN):", grid_knn.best_params_)
best_knn_params = grid_knn.best_params_
best_knn_model = grid_knn.best_estimator_

print("Best CV score (mean f1):", round(grid_knn.best_score_, 4))

y_kfold_pred_knn = best_knn_model.predict(X_A_test_processed)
y_kfold_prob_knn = best_knn_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_knn, zero_division=0, digits = 4))
auc_knn_kfold = roc_auc_score(y_A_test, y_kfold_prob_knn)
print(f"AUC-ROC: {auc_knn_kfold:.4f}")

Start training process for KNN
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters (KNN): {'n_neighbors': 3}
Best CV score (mean f1): 0.8565

Classification Report:
              precision    recall  f1-score   support

           0     0.9944    0.8512    0.9172     77864
           1     0.1195    0.8074    0.2081      1947

    accuracy                         0.8501     79811
   macro avg     0.5569    0.8293    0.5627     79811
weighted avg     0.9730    0.8501    0.8999     79811

AUC-ROC: 0.8600


In [17]:
xgb_base = XGBClassifier(eval_metric='auc', random_state=42, scale_pos_weight=neg/pos)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7, 10],
    'learning_rate': [0.05, 0.1],
}

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for XGBoost")
grid_xgb.fit(X_A_resampled, y_A_resampled)

print("Best parameters (XGBoost):", grid_xgb.best_params_)
best_xgb_params = grid_xgb.best_params_
best_xgb_model = grid_xgb.best_estimator_

print("Best CV score (mean f1):", round(grid_xgb.best_score_, 4))

y_kfold_pred_xgb= best_xgb_model.predict(X_A_test_processed)
y_kfold_prob_xgb = best_xgb_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_xgb, zero_division=0, digits = 4))
auc_xgb_kfold = roc_auc_score(y_A_test, y_kfold_prob_xgb)
print(f"AUC-ROC: {auc_xgb_kfold:.4f}")


Start training process for XGBoost
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters (XGBoost): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200}
Best CV score (mean f1): 0.954

Classification Report:
              precision    recall  f1-score   support

           0     0.9944    0.9440    0.9685     77864
           1     0.2600    0.7863    0.3908      1947

    accuracy                         0.9402     79811
   macro avg     0.6272    0.8652    0.6797     79811
weighted avg     0.9765    0.9402    0.9545     79811

AUC-ROC: 0.9400


In [18]:
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight

lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_A_resampled, y_A_resampled)

print("Best parameters (LightGBM):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_kfold_pred_lgbm= best_lgbm_model.predict(X_A_test_processed)
y_kfold_prob_lgbm = best_lgbm_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_lgbm, zero_division=0, digits = 4))
auc_lgbm_kfold = roc_auc_score(y_A_test, y_kfold_prob_lgbm)
print(f"AUC-ROC: {auc_lgbm_kfold:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

[LightGBM] [Info] Number of positive: 51910, number of negative: 103818
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.728177 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37601
[LightGBM] [Info] Number of data points in the train set: 155728, number of used features: 1009
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333338 -> initscore=-0.693128
[LightGBM] [Info] Start training from score -0.693128
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Info] Total Bins 37601
[LightGBM] [Info] Number of data points in the train set: 155728, number of used features: 1009
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333338 -> initscore=-0.693128
[LightGBM] [Info] Start training from score -0.693128
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

...


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

[LightGBM] [Info] Number of positive: 77864, number of negative: 155728
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.690228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41378
[LightGBM] [Info] Number of data points in the train set: 233592, number of used features: 1080
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Best parameters (LightGBM): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.973

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9909    0.9861    0.9885     77864
           1     0.5343    0.6364    0.5809      1947

    accuracy                         0.9776     79811
   macro avg     0.7626    0.8112    0.7847     79811
weighted avg     0.9797    0.9776    0.9785     79811

AUC-ROC: 0.9427


# Train trên A, test trên B

In [19]:
y_kfold_pred_lgbm_B= best_lgbm_model.predict(X_B_test_processed1)
y_kfold_prob_lgbm_B = best_lgbm_model.predict_proba(X_B_test_processed1)[:, 1]
print("\nClassification Report:")
print(classification_report(y_B_test, y_kfold_pred_lgbm_B, zero_division=0, digits = 4))
auc_lgbm_kfold_B = roc_auc_score(y_B_test, y_kfold_prob_lgbm_B)
print(f"AUC-ROC: {auc_lgbm_kfold_B:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9761    0.9976    0.9867    209737
           1     0.1937    0.0235    0.0419      5243

    accuracy                         0.9738    214980
   macro avg     0.5849    0.5105    0.5143    214980
weighted avg     0.9570    0.9738    0.9637    214980

AUC-ROC: 0.7288


# Train trên B, test trên A

In [20]:
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight

lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_B_resampled, y_B_resampled)

print("Best parameters (LightGBM):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_kfold_pred_lgbm_A= best_lgbm_model.predict(X_A_test_processed1)
y_kfold_prob_lgbm_A = best_lgbm_model.predict_proba(X_A_test_processed1)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_lgbm_A, zero_division=0, digits = 4))
auc_lgbm_kfold = roc_auc_score(y_A_test, y_kfold_prob_lgbm_A)
print(f"AUC-ROC: {auc_lgbm_kfold:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

...


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

[LightGBM] [Info] Number of positive: 209735, number of negative: 419470
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.060893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 38699
[LightGBM] [Info] Number of data points in the train set: 629205, number of used features: 1171
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
Best parameters (LightGBM): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.9672


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9782    0.9889    0.9835     77864
           1     0.2108    0.1186    0.1518      1947

    accuracy                         0.9677     79811
   macro avg     0.5945    0.5538    0.5677     79811
weighted avg     0.9595    0.9677    0.9632     79811

AUC-ROC: 0.6458


# Train trên A+B, kfold ###

In [21]:
X_raw_train = pd.concat([X_A_train, X_B_train]).reset_index(drop=True)
y_raw_train = pd.concat([y_A_train, y_B_train]).reset_index(drop=True)

X_raw_test = pd.concat([X_A_test, X_B_test]).reset_index(drop=True)
y_raw_test = pd.concat([y_A_test, y_B_test]).reset_index(drop=True)
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(sampling_strategy=0.25, random_state=42)),
    ('undersample', RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
])

X_resampled, y_resampled = pipeline.fit_resample(X_raw_train, y_raw_train)
X_test_processed = pipeline.named_steps['preprocess'].transform(X_raw_test)

In [22]:
lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_resampled, y_resampled)

print("Best parameters (XGBoost):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_pred= best_lgbm_model.predict(X_test_processed )
y_prob = best_lgbm_model.predict_proba(X_test_processed )[:, 1]
print("\nClassification Report:")
print(classification_report(y_raw_test, y_pred, zero_division=0, digits = 4))
auc = roc_auc_score(y_raw_test, y_prob)
print(f"AUC-ROC: {auc:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


[LightGBM] [Info] Number of data points in the train set: 419470, number of used features: 1158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333335 -> initscore=-0.693140
[LightGBM] [Info] Start training from score -0.693140
[LightGBM] [Info] Number of positive: 139824, number of negative: 279646
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 4.883554 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 40172
[LightGBM] [Info] Number of data points in the train set: 419470, number of used features: 1158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333335 -> initscore=-0.693140
[LightGBM] [Info] Start training from score -0.693140
[LightGBM] [Info] Number of positive: 139823, number of negative: 279647
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 7.249030 seconds.
You can set `force_row_wise=

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number

...


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

[LightGBM] [Info] Number of positive: 287599, number of negative: 575198
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.934973 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 40849
[LightGBM] [Info] Number of data points in the train set: 862797, number of used features: 1212
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
Best parameters (XGBoost): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.9592


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9919    0.9836    0.9877    287601
           1     0.5084    0.6783    0.5812      7190

    accuracy                         0.9762    294791
   macro avg     0.7501    0.8310    0.7845    294791
weighted avg     0.9801    0.9762    0.9778    294791

AUC-ROC: 0.9387


# Train trên A+KB, Test trên (1-K)B

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)
from sklearn.model_selection import train_test_split
from scipy.sparse import vstack

K_values = [30, 50, 70]

for K in K_values:
    print(f"\n=== K = {K}% ===")

    X_KB_raw, X_not_KB_raw, y_KB_raw, y_not_KB_raw = train_test_split(
        X_B_train, y_B_train,
        train_size=K/100,
        stratify=y_B_train,
        random_state=42
    )

    X_train_raw = pd.concat([X_A_train, X_KB_raw]).reset_index(drop=True)
    y_train_raw = pd.concat([y_A_train, y_KB_raw]).reset_index(drop=True)

    pipeline = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(sampling_strategy=0.25, random_state=42)),
        ('undersample', RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
        ('model', LGBMClassifier(
            objective='binary',
            random_state=42,
            learning_rate=0.1,
            max_depth=10,
            n_estimators=200,
            num_leaves=31
        ))
    ])

    pipeline.fit(X_train_raw, y_train_raw)

    y_pred = pipeline.predict(X_not_KB_raw)
    y_prob = pipeline.predict_proba(X_not_KB_raw)[:, 1]

    print("\nClassification Report:")
    print(classification_report(y_not_KB_raw, y_pred, zero_division=0, digits=4))

    auc = roc_auc_score(y_not_KB_raw, y_prob)
    print(f"AUC-ROC: {auc:.4f}")


=== K = 30% ===
[LightGBM] [Info] Number of positive: 140784, number of negative: 281568
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.382676 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 44505
[LightGBM] [Info] Number of data points in the train set: 422352, number of used features: 1181
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9909    0.9885    0.9897    587260
           1     0.5808    0.6349    0.6067     14682

    accuracy                         0.9799    601942
   macro avg     0.7858    0.8117    0.7982    601942
weighted avg     0.9809    0.9799    0.9804    601942

AUC-ROC: 0.9296

=== K = 50% ===
[LightGBM] [Info] Number of positive: 182731, number of negative: 365462
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.747766 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 43813
[LightGBM] [Info] Number of data points in the train set: 548193, number of used features: 1192
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9917    0.9877    0.9897    419472
           1     0.5760    0.6707    0.6198     10487

    accuracy                         0.9799    429959
   macro avg     0.7839    0.8292    0.8047    429959
weighted avg     0.9816    0.9799    0.9807    429959

AUC-ROC: 0.9377

=== K = 70% ===
[LightGBM] [Info] Number of positive: 224678, number of negative: 449356
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.287796 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41754
[LightGBM] [Info] Number of data points in the train set: 674034, number of used features: 1209
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Classification Report:
              precision    recall  f1-score   support

           0     0.9922    0.9867    0.9894    251684
           1     0.5646    0.6890    0.6206      6292

    accuracy                         0.9795    257976
   macro avg     0.7784    0.8378    0.8050    257976
weighted avg     0.9818    0.9795    0.9804    257976

AUC-ROC: 0.9414
